# A notebook for examining the symmetry of the key matrix on various MDPs

$$\begin{align}
e_t^\top S^2e_t  > -\frac{1}{2}e_t^\top (SK -K S) e_t\\
\end{align}
$$

#### MDP 1: a 3-chain which terminates at the end

In [1]:
import sys
if '..' not in sys.path:
    sys.path.insert(0, '..')

import jax.numpy as jnp
I = jnp.eye(4)
γ=0.99
P = jnp.array([
[0,1,0,0], 
[0,0,1,0],
[0,0,0,1],
[1,0,0,0], # terminal state
])


def compute_mu():
    rho_0 = jnp.zeros(4)
    rho_0 = rho_0.at[0].set(1.0)
    A = I - γ * P.T
    d_gamma = jnp.linalg.solve(A, (1 - γ) * rho_0)
    d_gamma_norm = d_gamma / jnp.sum(d_gamma)
    return d_gamma_norm
D = jnp.diag(compute_mu())


# def compute_d():
#     A = P.T - I
#     A = A.at[-1, :].set(1.0)
#     b = jnp.array([1,0,0,0])
#     mu = jnp.linalg.solve(A, b)
#     mu = jnp.clip(mu, a_min=0.0)
#     return mu / mu.sum()

# D = jnp.diag(compute_d()) # undiscounted


In [2]:
def compute_alignment_diff(D, P):
    A = D @ (I - γ * P)
    S = 0.5 * (A + A.T)
    K = 0.5 * (A - A.T)
    error = jnp.array([1,1,1,1])
    lhs = error.T @ (S**2) @ error
    rhs = -0.5 * error.T @ (S@K - K@S) @ error.T
    return lhs - rhs

def is_pd(D,P):
    A = D @ (I - γ * P)
    S = 0.5 * (A + A.T)
    K = 0.5 * (A - A.T)
    SA = S@A
    # 1. Extract the symmetric part of SA
    SA_symmetric = 0.5 * (SA + SA.T)
    # 2. Use 'eigvalsh' (the 'h' stands for Hermitian/Symmetric), which works perfectly on GPUs
    eigenvalues_SA = jnp.linalg.eigvalsh(SA_symmetric)
    # 3. Minimum eigenvalue
    min_eig_SA = jnp.min(eigenvalues_SA) # No jnp.real needed; symmetric eigenvalues are strictly real
    is_SA_pos_def = min_eig_SA > 0
    return is_SA_pos_def

    

In [3]:
is_pd(D,P)

Array(True, dtype=bool)

In [22]:
compute_alignment_diff(D,P)

Array(0.00053087, dtype=float32)

In [23]:
compute_alignment_diff(I,P)

Array(3.9802, dtype=float32)

In [24]:
is_pd(I,P)

Array(False, dtype=bool)

In [9]:
import numpy as np
# 3 state chain cycles
def cyclic_chain(N):
    """N-state deterministic cycle: 0->1->2->...->N-1->0"""
    P = np.zeros((N, N))
    for i in range(N):
        P[i, (i + 1) % N] = 1.0
    I = np.eye(N)
    return jnp.array(P), jnp.array(I)

N = 13
P, I = cyclic_chain(N)

gamma = 0.99

# 2. Compute the severely skewed Stationary Distribution (D)
A_mu = P.T - I
A_mu = A_mu.at[-1, :].set(1.0)
b = jnp.zeros(P.shape[0])
b = b.at[-1].set(1.0)
mu = jnp.linalg.solve(A_mu, b)
mu = jnp.clip(mu, a_min=0.0)
mu = mu / mu.sum()
D = jnp.diag(mu)

# 3. Compute A, S, and SA
A = D @ (I - gamma * P)
S = 0.5 * (A + A.T)
SA = S @ A

# 4. Extract the Symmetric Part of SA
SA_sym = 0.5 * (SA + SA.T)

# 5. Find the Eigenvalues and Eigenvectors of SA_sym
# eigh returns eigenvalues in ascending order, so index 0 is the minimum
eigenvalues, eigenvectors = jnp.linalg.eigh(SA_sym)
min_eig = eigenvalues[0]

print(f"Stationary Distribution (mu): {mu}")
print(f"Minimum Eigenvalue of SA_sym: {min_eig:.6f}\n")

if min_eig < 0:
    print("FATAL FLAW DETECTED: SA has a negative eigenvalue.")
    
    # 6. Extract the fatal error vector
    fatal_e = eigenvectors[:, 0]
    print(f"The Fatal Error Vector (e): {fatal_e}")
    
    # 7. Prove the Alignment Condition is Negative
    # Trace = e^T * SA * e
    alignment_trace = fatal_e.T @ SA @ fatal_e
    
    print(f"Alignment Condition (e^T SA e): {alignment_trace:.6f}")
else:
    print("Matrix is safe. SA is Positive Definite.")


Stationary Distribution (mu): [0.07692308 0.07692308 0.07692308 0.07692308 0.07692308 0.07692308
 0.07692308 0.07692308 0.07692308 0.07692308 0.07692308 0.07692308
 0.07692308]
Minimum Eigenvalue of SA_sym: 0.000001

Matrix is safe. SA is Positive Definite.


In [62]:
import jax.numpy as jnp
import numpy as np

# 1. Construct a Pathological Topology
# A 3-state continuing chain with a severe bottleneck at State 2.
# States 0 and 1 flush instantly. State 2 traps the agent 95% of the time.
P = jnp.array([
    [0.0, 1.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.45, 0.45, 0.1],
    [0, 1.0, 0, 0, 0], 
    [0, 1.0, 0, 0, 0], 
    [0.01, 0.90, 0, 0, 0],
])

gamma = 0.99
I = jnp.eye(P.shape[0])

# 2. Compute the severely skewed Stationary Distribution (D)
A_mu = P.T - I
A_mu = A_mu.at[-1, :].set(1.0)
b = jnp.zeros(P.shape[0])
b = b.at[-1].set(1.0)
mu = jnp.linalg.solve(A_mu, b)
mu = jnp.clip(mu, a_min=0.0)
mu = mu / mu.sum()
D = jnp.diag(mu)

# 3. Compute A, S, and SA
A = D @ (I - gamma * P)
S = 0.5 * (A + A.T)
SA = S @ A

# 4. Extract the Symmetric Part of SA
SA_sym = 0.5 * (SA + SA.T)

# 5. Find the Eigenvalues and Eigenvectors of SA_sym
# eigh returns eigenvalues in ascending order, so index 0 is the minimum
eigenvalues, eigenvectors = jnp.linalg.eigh(SA_sym)
min_eig = eigenvalues[0]

print(f"Stationary Distribution (mu): {mu}")
print(f"Minimum Eigenvalue of SA_sym: {min_eig:.6f}\n")

if min_eig < 0:
    print("FATAL FLAW DETECTED: SA has a negative eigenvalue.")
    
    # 6. Extract the fatal error vector
    fatal_e = eigenvectors[:, 0]
    print(f"The Fatal Error Vector (e): {fatal_e}")
    
    # 7. Prove the Alignment Condition is Negative
    # Trace = e^T * SA * e
    alignment_trace = fatal_e.T @ SA @ fatal_e
    
    print(f"Alignment Condition (e^T SA e): {alignment_trace:.6f}")
else:
    print("Matrix is safe. SA is Positive Definite.")

Stationary Distribution (mu): [0.00054645 0.49726778 0.22377048 0.22377048 0.05464482]
Minimum Eigenvalue of SA_sym: 0.000000

Matrix is safe. SA is Positive Definite.


In [59]:
A.shape

(5,)

In [41]:
e = jnp.array([0,-10, 0.1, -5])
e @ SA @ e

Array(-0.24499655, dtype=float32)

In [45]:
K = 0.5*(A- A.T)
K

Array([[ 0.        , -0.00494999,  0.        ,  0.        ],
       [ 0.00494999,  0.        , -0.00123128, -0.00123128],
       [ 0.        ,  0.00123128,  0.        ,  0.        ],
       [ 0.        ,  0.00123128,  0.        ,  0.        ]],      dtype=float32)

In [61]:
import jax.numpy as jnp
import numpy as np

# 1. Construct a Pathological Topology
# A 3-state continuing chain with a severe bottleneck at State 2.
# States 0 and 1 flush instantly. State 2 traps the agent 95% of the time.
P = jnp.array([
    [0.0, 1.0, 0.0],  # 0 goes strictly to 1
    [0.0, 0.0, 1.0],  # 1 goes strictly to 2
    [0.1, 0.0, 0.9],  # 2 usually stays in 2, rarely loops to 0
])

gamma = 0.99
I = jnp.eye(3)

# 2. Compute the severely skewed Stationary Distribution (D)
A_mu = P.T - I
A_mu = A_mu.at[-1, :].set(1.0)
b = jnp.zeros(P.shape[0])
b = b.at[-1].set(1.0)
mu = jnp.linalg.solve(A_mu, b)
mu = jnp.clip(mu, a_min=0.0)
mu = mu / mu.sum()
D = jnp.diag(mu)

# 3. Compute A, S, and SA
A = D @ (I - gamma * P)
S = 0.5 * (A + A.T)
SA = S @ A

# 4. Extract the Symmetric Part of SA
SA_sym = 0.5 * (SA + SA.T)

# 5. Find the Eigenvalues and Eigenvectors of SA_sym
# eigh returns eigenvalues in ascending order, so index 0 is the minimum
eigenvalues, eigenvectors = jnp.linalg.eigh(SA_sym)
min_eig = eigenvalues[0]

print(f"Stationary Distribution (mu): {mu}")
print(f"Minimum Eigenvalue of SA_sym: {min_eig:.6f}\n")

if min_eig < 0:
    print("FATAL FLAW DETECTED: SA has a negative eigenvalue.")
    
    # 6. Extract the fatal error vector
    fatal_e = eigenvectors[:, 0]
    print(f"The Fatal Error Vector (e): {fatal_e}")
    
    # 7. Prove the Alignment Condition is Negative
    # Trace = e^T * SA * e
    alignment_trace = fatal_e.T @ SA @ fatal_e
    
    print(f"Alignment Condition (e^T SA e): {alignment_trace:.6f}")
else:
    print("Matrix is safe. SA is Positive Definite.")

Stationary Distribution (mu): [0.08333334 0.08333334 0.8333333 ]
Minimum Eigenvalue of SA_sym: 0.000010

Matrix is safe. SA is Positive Definite.


In [86]:
import jax.numpy as jnp
import numpy as np

# 1. Construct a 7-State Irreversible Ring with a Bottleneck
N = 15
trap_prob = 0.99
P = jnp.zeros((N, N))
gamma = 0.99

I = jnp.eye(N)

# Build the MDP transition

# States 0 through 5 strictly push forward
for i in range(N - 1):
    P = P.at[i, i+1].set(1.0)
# State 6 traps the agent 99% of the time, looping back to 0 rarely
P = P.at[-1, -1].set(trap_prob)
P = P.at[-1, 0].set(1-trap_prob)

# 2. Compute the Undiscounted Stationary Distribution (Continuing Task)
A_mu = P.T - I
A_mu = A_mu.at[-1, :].set(1.0)
b = jnp.zeros(N)
b = b.at[-1].set(1.0)
mu = jnp.linalg.solve(A_mu, b)
mu = jnp.clip(mu, a_min=0.0)
mu = mu / mu.sum()
D = jnp.diag(mu)

# 3. Compute A, S, and SA
A = D @ (I - gamma * P)
S = 0.5 * (A + A.T)
SA = S @ A

# 4. Extract Symmetric Part and Eigenvalues
SA_sym = 0.5 * (SA + SA.T)
eigenvalues, eigenvectors = jnp.linalg.eigh(SA_sym)
min_eig = eigenvalues[0]

print(f"Stationary Distribution (mu):\n{np.round(mu, 4)}")
print(f"Minimum Eigenvalue of SA_sym: {min_eig:.4e}")

if min_eig < 0:
    print("FATAL FLAW DETECTED: SA has a negative eigenvalue.")
else:
    print("Matrix is safe. SA is Positive Definite.")

num_samples = 10_000_000

# Draw x ~ N(0, 1). We only need x^2 for the chi-squared trick
x_squared = np.random.randn(num_samples, N)**2

# Compute sum(lambda_i * x_i^2) for all samples simultaneously
# eigenvalues is shape (N,), x_squared is shape (samples, N)
quadratic_forms = np.sum(eigenvalues * x_squared, axis=1)

# Calculate the percentage of volume that falls below 0
danger_volume = np.mean(quadratic_forms < 0)

print(f"Percentage of error space causing divergence: {danger_volume * 100:.6f}%")

Stationary Distribution (mu):
[0.0088 0.0088 0.0088 0.0088 0.0088 0.0088 0.0088 0.0088 0.0088 0.0088
 0.0088 0.0088 0.0088 0.0088 0.8772]
Minimum Eigenvalue of SA_sym: -6.0916e-07
FATAL FLAW DETECTED: SA has a negative eigenvalue.
Percentage of error space causing divergence: 0.000000%


In [85]:
eigenvectors[0]

Array([ 0.41614583,  0.12485338,  0.24495651, -0.3027442 ,  0.28359574,
       -0.34399274,  0.23061258, -0.33528402,  0.1387507 , -0.28078178,
        0.04831609,  0.1866172 ,  0.00538665, -0.06540443,  0.3945428 ],      dtype=float32)

In [ ]:
# 5 chains from the start state